In [1]:
print("Ok, let's start with RAG demo")

Ok, let's start with RAG demo


Data Ingestion

In [ ]:
# Get file
from pathlib import Path
file_path = Path("../data/information.txt").resolve()
print(f"File path: {file_path}")

# Load document
from langchain_community.document_loaders import TextLoader
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()
print(f"Loaded {len(documents)} document(s)")

# Create chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks = text_splitter.split_documents(documents=documents)
print(f"Created {len(chunks)} chunk(s)")

from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv() # load environment variables from .env file
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

# Create vectorstore 
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(documents=chunks, embedding=embedding_model)

In [42]:
relevant_docs = vector_store.similarity_search("story of Captain Ahabâ€™s")

In [50]:
def format_relevant_docs(docs):
    return [doc for doc in docs]

In [ ]:
# Hybrid Search: BM25 + Vector Search
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Create BM25 retriever (keyword search)
bm25_retriever = BM25Retriever.from_documents(chunks)

# Create vector retriever
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# Combine both retrievers (ensemble)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)
print("✓ Hybrid retriever (BM25 + Vector)")

In [ ]:
# Reranking with Cohere
from langchain_cohere import CohereReranker
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

compressor = CohereReranker(model="rerank-english-v2.0")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever
)
print("✓ Reranker added (Cohere)")

In [ ]:
# Test Hybrid Search + Reranking
query = "story of Captain Ahab"
reranked_docs = compression_retriever.invoke(query)
print(f"Retrieved and reranked {len(reranked_docs)} document(s)")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\n{i}. {doc.page_content[:80]}...")

In [ ]:
def format_relevant_docs(docs):
    return [doc for doc in docs]

# Create chat prompt template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that answers questions based on the following context: {context}"), 
        ("user", "Question: {question}")
    ])

# Instantiate the model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Intantiate output parser
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

# Create RAG chain
rag_chain = prompt | llm | output_parser

# Invoke the RAG chain
context = format_relevant_docs(relevant_docs)
question = "What is the story of Captain Ahabâ€™s?" # Happy test case
# question = "What is the story of shubham?" # Negative test case

rag_chain.invoke(
    {"context": context, "question": question}
)